<a href="https://colab.research.google.com/github/Bravinkindi9/jolmi-research/blob/main/VV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================
# THRESHOLD SENSITIVITY SWEEP + RE-TIMED LATENCY
# Reporting configuration: VV polarization
# Run in Google Colab (Earth Engine already authenticated from your session)
# ============================================

import ee
ee.Authenticate()
ee.Initialize(project='sar-research-505215')

aoi = ee.Geometry.Rectangle([29.20, -1.75, 29.55, -1.50])

# ----------------------------------------------------------------
# CORRECTED: VV polarization (your script above used VH — mismatch fixed here)
# ----------------------------------------------------------------
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(aoi) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .select('VV')

beforeCollection = s1.filterDate('2023-04-15', '2023-04-30')
afterCollection  = s1.filterDate('2023-05-03', '2023-05-10')

print('Before images found:', beforeCollection.size().getInfo())
print('After images found:', afterCollection.size().getInfo())

postFloodImage = afterCollection.first()
acquisitionTime = postFloodImage.get('system:time_start')
print('Post-flood scene acquisition time:', ee.Date(acquisitionTime).getInfo())

before = beforeCollection.mean().clip(aoi)
after  = afterCollection.mean().clip(aoi)

beforeFiltered = before.focal_mean(50, 'circle', 'meters')
afterFiltered  = after.focal_mean(50, 'circle', 'meters')

diff = afterFiltered.subtract(beforeFiltered)

# ----------------------------------------------------------------
# PART 1 — Threshold sensitivity sweep (Table 1)
# Runs -2 through -6 dB, VV, prints hectares for each.
# Copy these numbers straight into supporting_tables.md.
# ----------------------------------------------------------------
print('\n=== THRESHOLD SENSITIVITY SWEEP (VV) ===')
thresholds = [-2, -3, -4, -5, -6]
sweep_results = {}

for t in thresholds:
    flooded_t = diff.lt(t).selfMask()
    area_ha = flooded_t.multiply(ee.Image.pixelArea()).divide(10000) \
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=aoi,
            scale=10,
            maxPixels=1e9
        ).get('VV').getInfo()
    sweep_results[t] = area_ha
    print(f'  {t} dB  ->  {area_ha:.2f} ha')

print('\nFull results dict (copy for your records):', sweep_results)

# ----------------------------------------------------------------
# PART 2 — Re-timed latency at -3 dB / VV (the actual reporting config)
# Run this cell 3 separate times, note each "Observed end-to-end
# script latency" line, then compute median + range yourself.
# ----------------------------------------------------------------
import datetime

print('\n=== LATENCY TIMING RUN (-3 dB, VV) ===')
startTime = datetime.datetime.now()
print('Processing started at:', startTime)

flooded_reporting = diff.lt(-3).selfMask()
floodedAreaHa = flooded_reporting.multiply(ee.Image.pixelArea()).divide(10000) \
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=10,
        maxPixels=1e9
    )

result = floodedAreaHa.getInfo()  # blocking call — this is what we time against
print('Estimated flooded area at -3 dB VV (hectares):', result)

endTime = datetime.datetime.now()
elapsedSeconds = (endTime - startTime).total_seconds()
print('Processing finished at:', endTime)
print('Observed end-to-end script latency (seconds):', elapsedSeconds)
print('Config: -3 dB threshold, VV polarization (reporting configuration)')

Before images found: 4
After images found: 4
Post-flood scene acquisition time: {'type': 'Date', 'value': 1683130855000}

=== THRESHOLD SENSITIVITY SWEEP (VV) ===
  -2 dB  ->  251.38 ha
  -3 dB  ->  88.94 ha
  -4 dB  ->  55.17 ha
  -5 dB  ->  38.38 ha
  -6 dB  ->  26.91 ha

Full results dict (copy for your records): {-2: 251.38415318401394, -3: 88.94162616641742, -4: 55.16572952575703, -5: 38.37567362136855, -6: 26.907600588226437}

=== LATENCY TIMING RUN (-3 dB, VV) ===
Processing started at: 2026-09-17 09:38:35.743471
Estimated flooded area at -3 dB VV (hectares): {'VV': 88.94162616641742}
Processing finished at: 2026-09-17 09:38:36.361638
Observed end-to-end script latency (seconds): 0.618167
Config: -3 dB threshold, VV polarization (reporting configuration)


In [5]:

# ----------------------------------------------------------------
# PART 2 — Re-timed latency at -3 dB / VV (the actual reporting config)
# Run this cell 3 separate times, note each "Observed end-to-end
# script latency" line, then compute median + range yourself.
# ----------------------------------------------------------------
import datetime

print('\n=== LATENCY TIMING RUN (-3 dB, VV) ===')
startTime = datetime.datetime.now()
print('Processing started at:', startTime)

flooded_reporting = diff.lt(-3).selfMask()
floodedAreaHa = flooded_reporting.multiply(ee.Image.pixelArea()).divide(10000) \
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=10,
        maxPixels=1e9
    )

result = floodedAreaHa.getInfo()  # blocking call — this is what we time against
print('Estimated flooded area at -3 dB VV (hectares):', result)

endTime = datetime.datetime.now()
elapsedSeconds = (endTime - startTime).total_seconds()
print('Processing finished at:', endTime)
print('Observed end-to-end script latency (seconds):', elapsedSeconds)
print('Config: -3 dB threshold, VV polarization (reporting configuration)')


=== LATENCY TIMING RUN (-3 dB, VV) ===
Processing started at: 2026-09-17 09:48:35.601314
Estimated flooded area at -3 dB VV (hectares): {'VV': 88.94162616641742}
Processing finished at: 2026-09-17 09:48:35.795444
Observed end-to-end script latency (seconds): 0.19413
Config: -3 dB threshold, VV polarization (reporting configuration)


In [ ]:

=== LATENCY TIMING RUN (-3 dB, VV) ===
Processing started at: 2026-09-17 09:44:51.886642
Estimated flooded area at -3 dB VV (hectares): {'VV': 88.94162616641742}
Processing finished at: 2026-09-17 09:44:52.075346
Observed end-to-end script latency (seconds): 0.188704
Config: -3 dB threshold, VV polarization (reporting configuration)

In [6]:
# ============================================
# OTSU AUTOMATIC THRESHOLDING TEST — VV polarization
# Confirms whether the Otsu failure holds under the corrected
# reporting config (previous Otsu tests may have run on VH).
# Run in the same Colab session, after the sweep/latency script.
# ============================================

import ee

# Uses `diff` already computed in your session (VV, before/after, smoothed).
# If running fresh, re-run the setup block from threshold_sweep_and_latency.py first.

# ----------------------------------------------------------------
# STEP 1 — Build the histogram Otsu needs to operate on
# ----------------------------------------------------------------
histogram = diff.reduceRegion(
    reducer=ee.Reducer.histogram(255, 2),
    geometry=aoi,
    scale=10,
    bestEffort=True
).get('VV')

hist_dict = histogram.getInfo()
print('Histogram retrieved. Bucket count:', len(hist_dict['histogram']))
print('Histogram min:', hist_dict['bucketMin'], ' bucket width:', hist_dict['bucketWidth'])

# ----------------------------------------------------------------
# STEP 2 — Run Otsu's method on that histogram
# This is the standard GEE Otsu implementation (variance-based,
# same algorithm referenced in the KTH thesis).
# ----------------------------------------------------------------
def otsu(histogram):
    counts = ee.Array(ee.Dictionary(histogram).get('histogram'))
    means = ee.Array(ee.Dictionary(histogram).get('bucketMeans')) if 'bucketMeans' in histogram else None
    # Standard GEE community Otsu implementation using counts + bucket edges
    total = counts.reduce(ee.Reducer.sum(), [0]).get([0])
    total = ee.Number(total)

    means_list = ee.List.sequence(
        ee.Number(histogram['bucketMin']),
        ee.Number(histogram['bucketMin']).add(
            ee.Number(histogram['bucketWidth']).multiply(len(histogram['histogram']) - 1)),
        histogram['bucketWidth']
    )
    means_arr = ee.Array(means_list)

    sum_total = means_arr.multiply(counts).reduce(ee.Reducer.sum(), [0]).get([0])
    sum_total = ee.Number(sum_total)

    def compute_bss(i):
        i = ee.Number(i)
        aCounts = counts.slice(0, 0, i)
        aCount = aCounts.reduce(ee.Reducer.sum(), [0]).get([0])
        aMeans = means_arr.slice(0, 0, i)
        aMean = ee.Number(
            aMeans.multiply(aCounts).reduce(ee.Reducer.sum(), [0]).get([0])
        ).divide(ee.Number(aCount).max(1))

        bCount = total.subtract(aCount)
        bMean = sum_total.subtract(ee.Number(aCount).multiply(aMean)).divide(bCount.max(1))

        return ee.Number(aCount).multiply(bCount).multiply(aMean.subtract(bMean).pow(2))

    indices = ee.List.sequence(1, len(histogram['histogram']) - 1)
    bss = indices.map(compute_bss)

    # Threshold = bucket mean at the index of max between-class variance
    max_bss_index = ee.Array(bss).argmax().get(0)
    threshold = means_arr.get([max_bss_index])
    return threshold

otsu_threshold = otsu(hist_dict)
otsu_threshold_value = otsu_threshold.getInfo()
print('\n=== OTSU RESULT (VV) ===')
print('Otsu-selected threshold (dB):', otsu_threshold_value)
print('Manually chosen threshold for comparison (dB):', -3)

# ----------------------------------------------------------------
# STEP 3 — Apply the Otsu threshold and compute resulting flood area,
# to see whether it produces a plausible or implausible result.
# ----------------------------------------------------------------
flooded_otsu = diff.lt(otsu_threshold_value).selfMask()
otsu_area_ha = flooded_otsu.multiply(ee.Image.pixelArea()).divide(10000) \
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=10,
        maxPixels=1e9
    ).get('VV').getInfo()

print('Flooded area using Otsu threshold (ha):', otsu_area_ha)
print('Flooded area using manual -3 dB threshold (ha): 88.94')
print('\nCompare these two numbers directly — this tells us whether Otsu')
print('still fails (wildly different/implausible area) under VV, or whether')
print('the earlier failure was specific to the VH run.')

Histogram retrieved. Bucket count: 13
Histogram min: -14  bucket width: 2

=== OTSU RESULT (VV) ===
Otsu-selected threshold (dB): -2
Manually chosen threshold for comparison (dB): -3
Flooded area using Otsu threshold (ha): 251.38415318401394
Flooded area using manual -3 dB threshold (ha): 88.94

Compare these two numbers directly — this tells us whether Otsu
still fails (wildly different/implausible area) under VV, or whether
the earlier failure was specific to the VH run.


In [11]:
# ============================================
# FLOOD EXTENT MAP — VV corrected, reporting config (-3 dB)
# Run in the same Colab session (reuses diff, aoi, before, after)
# ============================================


import ee
import geemap
from geemap import cartoee
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs

# Reporting threshold, VV — matches your locked result (88.94 ha)
flooded = diff.lt(-3).selfMask()

region = aoi.bounds().getInfo()['coordinates']

fig = plt.figure(figsize=(12, 10))

# Basemap context: pre-flood VV backscatter in greyscale, so the reader can
# see actual terrain texture underneath the flood overlay (not a blank canvas)
ax = cartoee.get_map(before, region=region, vis_params={'min': -25, 'max': 0, 'palette': ['black', 'white']})

# Flood extent overlay, bold and distinct
cartoee.add_layer(ax, flooded, region=region, vis_params={'palette': ['#00BFFF']})

cartoee.add_gridlines(ax, interval=[0.1, 0.1], linestyle=':')
cartoee.add_north_arrow(ax, text='N', xy=(0.92, 0.92), text_color='black',
                         arrow_color='black', fontsize=16)
cartoee.add_scale_bar_lite(ax, length=10, xy=(0.05, 0.05), fontsize=10,
                            color='black', unit='km')

legend_patches = [
    mpatches.Patch(color='#00BFFF', label='Detected flood extent (−3 dB, VV)'),
]
ax.legend(handles=legend_patches, loc='lower left', fontsize=9, framealpha=0.9)

ax.set_title('Sentinel-1 SAR-Detected Flood Extent\nNyabihu/Ngororero, Rwanda — May 3, 2023\n'
             '88.94 ha at −3 dB threshold, VV polarization',
             fontsize=13, fontweight='bold')

fig.text(0.5, 0.01,
         'Basemap: pre-flood Sentinel-1 VV backscatter (greyscale). '
         'Data: Copernicus Sentinel-1 GRD, ESA, via Google Earth Engine.',
         ha='center', fontsize=8, style='italic')

plt.tight_layout()
plt.savefig('flood_extent_map_vv.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved as flood_extent_map_vv.png — download from the Colab file panel.")

NameError: name 'ccrs' is not defined

<Figure size 1200x1000 with 0 Axes>

In [8]:
!pip install cartopy
import cartopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 61.4 MB/s eta 0:00:00
